In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

PRED_PATH = "Data/predictions_ff3_vs_nn_2024_standard_retrain.parquet"  # change if needed
TARGET = "excess_ret"
BENCH_COL = "ff3_pred"
CAND_COL  = "nn_pred"


FILE: Data/predictions_ff3_vs_nn_2024_standard_retrain.parquet
Loaded preds: (684286, 5)
Date range: 2024-01-02 to 2024-12-31
Permnos: 2716


In [ ]:
preds = pd.read_parquet(PRED_PATH)
preds["date"] = pd.to_datetime(preds["date"])

required = {"date", "permno", TARGET, BENCH_COL, CAND_COL}
missing = required - set(preds.columns)
if missing:
    raise ValueError(f"Missing columns in preds: {missing}")

preds = preds.dropna(subset=[TARGET, BENCH_COL, CAND_COL, "permno", "date"]).copy()

print("FILE:", PRED_PATH)
print("Loaded preds:", preds.shape)
print("Date range:", preds["date"].min().date(), "to", preds["date"].max().date())
print("Permnos:", preds["permno"].nunique())

In [ ]:
preds["e_ff3"]  = preds[TARGET].astype(float) - preds[BENCH_COL].astype(float)
preds["e_nn"]   = preds[TARGET].astype(float) - preds[CAND_COL].astype(float)
preds["se_ff3"] = preds["e_ff3"] ** 2
preds["se_nn"]  = preds["e_nn"] ** 2

# Loss differential for testing: benchmark - candidate
# Positive -> candidate (NN) has lower squared error
preds["d"] = preds["se_ff3"] - preds["se_nn"]

mspe_ff3 = float(preds["se_ff3"].mean())
mspe_nn  = float(preds["se_nn"].mean())
delta_mspe = mspe_nn - mspe_ff3
rel_gap = delta_mspe / mspe_ff3

print("\nMSPE (pooled over all firm-day observations):")
print("FF3 MSPE:", mspe_ff3)
print("NN  MSPE:", mspe_nn)
print("Delta (NN - FF3):", delta_mspe)
print("Relative gap (NN/FF3 - 1):", rel_gap)


MSPE (pooled over all firm-day observations):
FF3 MSPE: 0.001920165841881584
NN  MSPE: 0.0019269391375083024
Delta (NN - FF3): 6.773295626718483e-06
Relative gap (NN/FF3 - 1): 0.003527453451667114


In [ ]:
df = preds.dropna(subset=["d", "permno", "date"]).copy()

# pass integer-coded clusters to avoid statsmodels dtype issues
df["g_permno"] = pd.Categorical(df["permno"]).codes
df["g_date"]   = pd.Categorical(df["date"]).codes

y = df["d"].to_numpy(dtype=float)
X = np.ones((len(df), 1))

res = sm.OLS(y, X).fit(
    cov_type="cluster",
    cov_kwds={"groups": (df["g_permno"].to_numpy(), df["g_date"].to_numpy())}
)

alpha = float(res.params[0])   # mean loss differential (FF3 - NN)
se    = float(res.bse[0])
tstat = float(res.tvalues[0])

# One-sided: H1 alpha > 0  <=> NN improves (lower MSE)
p_one = float(1 - stats.norm.cdf(tstat))
p_two = float(2 * (1 - stats.norm.cdf(abs(tstat))))

print("\nTwo-way clustered panel test: d_it = alpha + u_it")
print(f"alpha (mean d = MSPE_FF3 - MSPE_NN): {alpha:.8g}")
print(f"SE (2-way cluster):                 {se:.8g}")
print(f"t-stat:                             {tstat:.4f}")
print(f"p (one-sided, H1: NN improves):     {p_one:.4g}")
print(f"p (two-sided):                      {p_two:.4g}")


Two-way clustered panel test: d_it = alpha + u_it
alpha (mean d = MSPE_FF3 - MSPE_NN): -6.7732956e-06
SE (2-way cluster):                 1.1867411e-06
t-stat:                             -5.7075
p (one-sided, H1: NN improves):     1
p (two-sided):                      1.147e-08
